In [1]:
import pandas as pd
import numpy as np
import matplotlib as mpl
import matplotlib.pyplot as plt
from sklearn.preprocessing import StandardScaler

# Loading Data and Exploration

In [7]:
df = pd.read_csv("dataset_small__(11_nodes).csv")

FileNotFoundError: [Errno 2] No such file or directory: 'dataset_small__(11_nodes).csv'

In [ ]:
df.dtypes
df.describe()
#df.info()
#df.head()

In [ ]:
df.hist(bins = 30, figsize = (15,15))

In [ ]:
corr_matrix = df.corr().drop('travel_time_mins')
corr_matrix['travel_time_mins'].sort_values(ascending = False).round(3)

## Preprocessing

In [ ]:
X = df.drop('travel_time_mins',axis = 1)
y = df['travel_time_mins']

In [ ]:
missing = X.isnull().any().sum()
if missing > 0:
    X = X.fillna(X.median(),inplace = True)
    print(f"Filled with columns median value")
else:
    print(f"No empty cell found")

In [ ]:
from sklearn.preprocessing import StandardScaler
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)
X_scaled = pd.DataFrame(X_scaled,columns = X.columns)

# Split Data into Train and Test

In [ ]:
from sklearn.model_selection import train_test_split

X_train,X_test, y_train,y_test = train_test_split(X_scaled,y,test_size=0.2, random_state = 42)

# Training Model (Simple Linear Regression)

In [ ]:
from sklearn.linear_model import LinearRegression

lin_reg = LinearRegression()
lin_reg.fit(X_train, y_train)

##### Testing on some random values.

In [ ]:
random_x = X_train.iloc[:5]
random_y = y_train.iloc[:5]
print(f"Predictions:{lin_reg.predict(random_x).round(2)}")
print(f"Actual:     {list(random_y)}")

# Evaluation of the Model

In [ ]:
from sklearn.model_selection import cross_val_score
from sklearn.metrics import mean_absolute_error, r2_score

In [ ]:
y_pred = lin_reg.predict(X_test)
 
mae  = mean_absolute_error(y_test, y_pred)
rmse = np.sqrt(mean_squared_error(y_test, y_pred))
r2   = r2_score(y_test, y_pred)

print(f"\n  MAE  (Mean Absolute Error)       : {mae:.4f} minutes")
print(f"       → predictions off by {mae:.2f} mins on average")
print(f"\n  RMSE (Root Mean Squared Error)   : {rmse:.4f} minutes")
print(f"       → large errors penalised, {rmse:.2f} mins spread")
print(f"\n  R²   (Coefficient of Determination): {r2:.4f}")


## Cross Validation

In [ ]:
# Cross-validation score (more robust evaluation)
from sklearn.model_selection import cross_val_score

cv_scores = cross_val_score(lin_reg, X_train, y_train,cv=4, scoring="r2")

print(f"\n  4-Fold Cross-Validation R²:")
print(f"    Scores : {[round(s, 4) for s in cv_scores]}")
print(f"    Mean   : {cv_scores.mean():.4f}")
print(f"    Std    : {cv_scores.std():.4f}")

In [ ]:
print("\n" + "-"*60)
print("STEP 7: Visualising Results")
print("-"*60)
 
fig, ax = plt.subplots(1, 3, figsize=(15, 5))
fig.suptitle("Linear Regression — Model Evaluation", fontsize=14, fontweight="bold")
 
# Plot 1: Actual vs Predicted
ax[0].scatter(y_test, y_pred, alpha=0.3, s=10, color="blue")
min_val = min(y_test.min(), y_pred.min())
max_val = max(y_test.max(), y_pred.max())
ax[0].plot([min_val, max_val], [min_val, max_val], "r--", lw=1.5, label="Perfect prediction")
ax[0].set_xlabel("Actual travel time (mins)")
ax[0].set_ylabel("Predicted travel time (mins)")
ax[0].set_title(f"Actual vs Predicted\nR² = {r2:.4f}")
ax[0].legend(fontsize=8)
 
# Plot 2: Residuals (errors)
residuals = y_test - y_pred
ax[1].scatter(y_pred, residuals, alpha=0.3, s=10, color="brown")
ax[1].axhline(0, color="red", linestyle="--", lw=1.5)
ax[1].set_xlabel("Predicted travel time (mins)")
ax[1].set_ylabel("Residual (actual − predicted)")
ax[1].set_title(f"Residual Plot\nMAE = {mae:.4f} mins")
 
# Plot 3: Residual distribution
ax[2].hist(residuals, bins=40, color="green", alpha=0.75, edgecolor="white")
ax[2].axvline(0, color="black", linestyle="--", lw=1.5)
ax[2].set_xlabel("Residual (mins)")
ax[2].set_ylabel("Frequency")
ax[2].set_title(f"Residual Distribution\nRMSE = {rmse:.4f} mins")
 
plt.tight_layout()
plt.savefig("linear_regression_evaluation.png", dpi=150, bbox_inches="tight")
print("  Saved: linear_regression_evaluation.png")
plt.show()
